In [ ]:
import numpy as np
import scipy.stats as stats
import heapq

def confidence_interval(samples, confidence=0.95):
    """Return (mean, lower, upper) for a t-based CI of the sample mean."""
    samples = np.asarray(samples)
    n = samples.size
    mean = samples.mean()
    se = samples.std(ddof=1) / np.sqrt(n)

    alpha = 1 - confidence
    t_crit = stats.t.ppf(1 - alpha / 2, df=n - 1)

    theta_hat, ci_lower, ci_upper = mean, mean - t_crit * se, mean + t_crit * se

    return theta_hat, ci_lower, ci_upper

ARRIVAL, DEPARTURE = 0, 1

def simulate_blocking_cv(m, mean_service_time, mean_interarrival_time, num_customers):
    """DES of an M/M/m/m loss system (same logic as Ex 4).

    Returns the blocking fraction together with two auxiliary statistics
    that have known expectations and serve as control variates:
      Z1 = realised mean inter-arrival time            (E[Z1] = mean_interarrival_time)
      Z2 = realised mean service time of accepted customers (E[Z2] = mean_service_time)
    """
    busy_servers = 0
    blocked = 0
    arrived = 0
    accepted = 0
    sum_interarrival = 0.0
    sum_service = 0.0

    events = []
    ia0 = np.random.exponential(mean_interarrival_time)
    sum_interarrival += ia0
    heapq.heappush(events, (ia0, ARRIVAL))

    while arrived < num_customers:
        time, etype = heapq.heappop(events)
        if etype == ARRIVAL:
            arrived += 1
            if busy_servers < m:
                busy_servers += 1
                accepted += 1
                s = np.random.exponential(mean_service_time)
                sum_service += s
                heapq.heappush(events, (time + s, DEPARTURE))
            else:
                blocked += 1
            if arrived < num_customers:
                ia = np.random.exponential(mean_interarrival_time)
                sum_interarrival += ia
                heapq.heappush(events, (time + ia, ARRIVAL))
        else:
            busy_servers -= 1

    blocking_fraction = blocked / num_customers
    mean_interarrival_realized = sum_interarrival / num_customers
    mean_service_realized = sum_service / accepted
    return blocking_fraction, mean_interarrival_realized, mean_service_realized

In [35]:
# Ex 1
np.random.seed(42)

n = 100

U = np.random.uniform(0, 1, size=n)

samples = np.exp(U)

theta_hat, ci_lower, ci_upper = confidence_interval(samples)

print(f"Point estimate:        {theta_hat:.4f}")
print(f"Sample std dev:        {samples.std(ddof=1):.4f}")
print(f"95% CI:                ({ci_lower:.4f}, {ci_upper:.4f})")


Point estimate:        1.6721
Sample std dev:        0.5001
95% CI:                (1.5728, 1.7713)


In [36]:
# Ex 2
np.random.seed(42)

n = 100

U = np.random.uniform(0, 1, size=n)

xi = np.exp(U)
xiA= np.exp(1-U)
Yi = (xi+xiA)/2

theta_hat=1/n * Yi.sum()

theta_hat, ci_lower, ci_upper = confidence_interval(Yi)

print(f"Point estimate:        {theta_hat:.4f}")
print(f"Sample std dev:        {Yi.std(ddof=1):.4f}")
print(f"95% CI:                ({ci_lower:.4f}, {ci_upper:.4f})")


Point estimate:        1.7226
Sample std dev:        0.0626
95% CI:                (1.7102, 1.7350)


In [37]:
# Ex 3
np.random.seed(42)

xi = np.exp(U)

Zi = U

c = -(np.cov(xi,Zi)/(Zi.var()))

c = -np.cov(xi, Zi, ddof=1)[0, 1] / Zi.var(ddof=1)
yi = xi + c * (Zi - Zi.mean())

theta_hat, ci_lower, ci_upper = confidence_interval(yi)

print(f"Point estimate:        {theta_hat:.4f}")
print(f"Sample std dev:        {yi.std(ddof=1):.4f}")
print(f"95% CI:                ({ci_lower:.4f}, {ci_upper:.4f})")


Point estimate:        1.6721
Sample std dev:        0.0621
95% CI:                (1.6597, 1.6844)


In [45]:
# Ex 4
np.random.seed(42)

strata_count = 10
observations = 10

y = np.empty(strata_count)
var = np.empty(strata_count)

strata_lb = np.arange(0, 1, 1 / strata_count)
strata_ub = np.arange(1 / strata_count, 1 + 1 / strata_count, 1 / strata_count)

for i in range(strata_count):
    strata_samples = np.random.uniform(strata_lb[i], strata_ub[i], size=observations)
    strata_exp = np.exp(strata_samples)
    y[i] = strata_exp.mean()
    var[i] = strata_exp.var(ddof=1)

weights = np.full(strata_count, 1 / strata_count)
theta_hat = np.sum(weights * y)

# standard error for stratified estimator: sum_h w_h^2 * S_h^2 / n_h
se = np.sqrt(np.sum((weights ** 2) * var / observations))

t_crit = stats.t.ppf(1 - 0.95 / 2, df=strata_count - 1)
ci_lower = theta_hat - t_crit * se
ci_upper = theta_hat + t_crit * se

print(f"Point estimate:        {theta_hat:.4f}")
print(f"Estimator std error:   {se:.4f}")
print(f"95% CI:                ({ci_lower:.4f}, {ci_upper:.4f})")

Point estimate:        1.7137
Estimator std error:   0.0054
95% CI:                (1.7133, 1.7140)


In [ ]:
# Ex 5
np.random.seed(42)

m = 10
mean_service_time = 8.0
mean_interarrival_time = 1.0
num_customers = 10000
num_runs = 10

X = np.empty(num_runs)   # blocking fraction (crude estimator, as in Ex 4)
Z1 = np.empty(num_runs)  # control variate: mean inter-arrival time, E[Z1] = 1.0
Z2 = np.empty(num_runs)  # control variate: mean service time (accepted), E[Z2] = 8.0

for i in range(num_runs):
    X[i], Z1[i], Z2[i] = simulate_blocking_cv(m, mean_service_time, mean_interarrival_time, num_customers)

theta_crude, ci_lower, ci_upper = confidence_interval(X)

print("Crude estimator (Ex 4):")
print(f"  Point estimate: {theta_crude:.5f}")
print(f"  Sample std dev: {X.std(ddof=1):.5f}")
print(f"  95% CI:         ({ci_lower:.5f}, {ci_upper:.5f})")

Crude estimator (Ex 4):
  Point estimate: 0.11853
  Sample std dev: 0.00600
  95% CI:         (0.11424, 0.12282)


In [48]:
# Single control variate: realised mean service time (Z2, E[Z2] = 8.0)
c2 = -np.cov(X, Z2, ddof=1)[0, 1] / Z2.var(ddof=1)
Y2 = X + c2 * (Z2 - mean_service_time)

theta_cv2, ci_lower2, ci_upper2 = confidence_interval(Y2)

print(f"corr(X, Z2) = {np.corrcoef(X, Z2)[0, 1]:.4f}")
print(f"c           = {c2:.4f}\n")
print(f"Point estimate: {theta_cv2:.5f}")
print(f"Sample std dev: {Y2.std(ddof=1):.5f}")
print(f"95% CI:         ({ci_lower2:.5f}, {ci_upper2:.5f})")
print(f"Variance reduction vs crude: {1 - Y2.var(ddof=1) / X.var(ddof=1):.1%}")

corr(X, Z2) = 0.7932
c           = -0.0427

Point estimate: 0.12034
Sample std dev: 0.00365
95% CI:         (0.11772, 0.12295)
Variance reduction vs crude: 62.9%


In [40]:
# Combined control variates: Z1 (inter-arrival) and Z2 (service time)
Zc = np.column_stack([Z1 - mean_interarrival_time, Z2 - mean_service_time])
c_vec = np.linalg.lstsq(Zc, X - X.mean(), rcond=None)[0]
Y3 = X - Zc @ c_vec

theta_cv3, ci_lower3, ci_upper3 = confidence_interval(Y3)

print(f"c = (c1, c2) = ({c_vec[0]:.4f}, {c_vec[1]:.4f})\n")
print(f"Point estimate: {theta_cv3:.5f}")
print(f"Sample std dev: {Y3.std(ddof=1):.5f}")
print(f"95% CI:         ({ci_lower3:.5f}, {ci_upper3:.5f})")
print(f"Variance reduction vs crude: {1 - Y3.var(ddof=1) / X.var(ddof=1):.1%}")
print(f"\nErlang B (analytical) = 0.12166")

c = (c1, c2) = (-0.1625, 0.0358)

Point estimate: 0.12076
Sample std dev: 0.00269
95% CI:         (0.11884, 0.12269)
Variance reduction vs crude: 79.8%

Erlang B (analytical) = 0.12166


### Discussion

- The **crude estimator** from Ex 4 has a sample std. dev. of $\approx 0.0060$ over the 10 replications, giving a 95% CI of width $\approx 0.0086$.
- Using the **realised mean service time** $Z_2$ alone as a control variate is very effective: it is strongly correlated with the blocking fraction ($\rho \approx 0.79$) because replications where the (random) service-time draws happen to be longer keep servers occupied longer and therefore block more customers. This single control variate reduces the variance by **~63%**.
- **Combining** $Z_1$ (mean inter-arrival time) and $Z_2$ (mean service time) via a two-variable linear regression pushes the variance reduction to **~80%**, roughly halving the CI width compared to the crude estimator.
- All three estimators remain unbiased and their 95% CIs comfortably contain the analytical Erlang-B value $B(8,10) \approx 0.12166$, but the control-variate estimators get there with a substantially tighter interval -- i.e. the *same* number of simulation runs now yields a much more precise estimate of the blocking probability.

## Ex 7, Importance sampling for tail probabilities: $P(Z>a)$, $Z\sim N(0,1)$

We want to estimate $\theta(a) = P(Z>a)$ for $Z\sim N(0,1)$, for moderately large $a$ (e.g. $a=2,4$), where $\theta(a)$ becomes a *rare-event* probability ($\theta(2)\approx 0.0228$, $\theta(4)\approx 3.17\times10^{-5}$).

**Crude Monte Carlo**: draw $Z_i\sim N(0,1)$ and use $\hat\theta_{MC}=\frac{1}{n}\sum_i \mathbb{1}\{Z_i>a\}$. This is unbiased with variance $\theta(a)(1-\theta(a))/n$ — for small $\theta(a)$ almost every draw contributes 0, so the *relative* error $\sqrt{\mathrm{Var}}/\theta(a)$ is huge unless $n\gg 1/\theta(a)$.

**Importance sampling**: instead sample $Y_i\sim N(a,\sigma^2)$, which concentrates draws near the region $\{Y>a\}$ that actually matters, and reweight by the likelihood ratio (Radon-Nikodym derivative)
$$w(y)=\frac{f(y)}{g(y)}=\frac{\varphi(y;0,1)}{\varphi(y;a,\sigma^2)}=\sigma\,\exp\!\left(-\frac{y^2}{2}+\frac{(y-a)^2}{2\sigma^2}\right).$$
The IS estimator $\hat\theta_{IS}=\frac{1}{n}\sum_i \mathbb{1}\{Y_i>a\}\,w(Y_i)$ is unbiased for $\theta(a)$ for *any* $\sigma^2>0$, but its variance depends strongly on $\sigma^2$ and on how well the proposal matches the optimal (zero-variance) proposal, which is the law of $Z\mid Z>a$.

In [63]:
# Ex 7 - crude Monte Carlo vs importance sampling for P(Z > a), Z ~ N(0,1)
np.random.seed(42)

def true_prob(a):
    """Exact value of P(Z > a) for Z ~ N(0,1)."""
    return stats.norm.sf(a)

def crude_mc(a, n):
    """Crude MC estimator: theta_hat = mean(1{Z_i > a}), Z_i ~ N(0,1)."""
    Z = np.random.normal(0, 1, size=n)
    X = (Z > a).astype(float)
    return confidence_interval(X), X.var(ddof=1)

def importance_sampling(a, sigma2, n):
    """IS estimator with proposal Y_i ~ N(a, sigma2) and weights
    w(y) = phi(y; 0, 1) / phi(y; a, sigma2)."""
    sigma = np.sqrt(sigma2)
    Y = np.random.normal(a, sigma, size=n)
    w = sigma * np.exp(-Y**2 / 2 + (Y - a) ** 2 / (2 * sigma2))
    X = (Y > a).astype(float) * w
    return confidence_interval(X), X.var(ddof=1)

n = 10000
for a in (2, 4):
    (theta_mc, lo_mc, hi_mc), var_mc = crude_mc(a, n)
    (theta_is, lo_is, hi_is), var_is = importance_sampling(a, 1.0, n)
    print(f"a = {a}, true P(Z>a) = {true_prob(a):.6}")
    print(f"  Crude MC:    theta_hat = {theta_mc:.6}, var = {var_mc:.3}, 95% CI = ({lo_mc:.3}, {hi_mc:.3})")
    print(f"  IS (sig2=1): theta_hat = {theta_is:.6}, var = {var_is:.3}, 95% CI = ({lo_is:.3}, {hi_is:.3})")
    if var_mc > 0:
        print(f"  Variance reduction: {1 - var_is / var_mc:.4%}")
    else:
        print(f"  Variance reduction: n/a (crude variance = 0, zero hits in {n} draws)")
    print()

a = 2, true P(Z>a) = 0.0227501
  Crude MC:    theta_hat = 0.0237, var = 0.0231, 95% CI = (0.0207, 0.0267)
  IS (sig2=1): theta_hat = 0.0232753, var = 0.00126, 95% CI = (0.0226, 0.024)
  Variance reduction: 94.5584%

a = 4, true P(Z>a) = 3.16712e-05
  Crude MC:    theta_hat = 0.0, var = 0.0, 95% CI = (0.0, 0.0)
  IS (sig2=1): theta_hat = 3.10538e-05, var = 4.4e-09, 95% CI = (2.98e-05, 3.24e-05)
  Variance reduction: n/a (crude variance = 0, zero hits in 10000 draws)



In [65]:
# Effect of sample size on crude MC vs importance sampling (sigma^2 = 1)
sample_sizes = [100, 1000, 10000, 100000]

for a in (2, 4):
    print(f"a = {a}, true P(Z>a) = {true_prob(a):.6}")
    for n in sample_sizes:
        np.random.seed(42)
        (theta_mc, lo_mc, hi_mc), _ = crude_mc(a, n)
        np.random.seed(42)
        (theta_is, lo_is, hi_is), _ = importance_sampling(a, 1.0, n)
        print(f"  n={n:>7}: crude = {theta_mc:.3} (CI width {hi_mc - lo_mc:.3}),  "
              f"IS = {theta_is:.3} (CI width {hi_is - lo_is:.3})")
    print()

a = 2, true P(Z>a) = 0.0227501
  n=    100: crude = 0.0 (CI width 0.0),  IS = 0.0237 (CI width 0.0143)
  n=   1000: crude = 0.024 (CI width 0.019),  IS = 0.0241 (CI width 0.00445)
  n=  10000: crude = 0.0237 (CI width 0.00596),  IS = 0.0228 (CI width 0.00137)
  n= 100000: crude = 0.0227 (CI width 0.00185),  IS = 0.0228 (CI width 0.000433)

a = 4, true P(Z>a) = 3.16712e-05
  n=    100: crude = 0.0 (CI width 0.0),  IS = 3.38e-05 (CI width 2.69e-05)
  n=   1000: crude = 0.0 (CI width 0.0),  IS = 3.41e-05 (CI width 8.61e-06)
  n=  10000: crude = 0.0 (CI width 0.0),  IS = 3.17e-05 (CI width 2.64e-06)
  n= 100000: crude = 1e-05 (CI width 3.92e-05),  IS = 3.19e-05 (CI width 8.39e-07)



In [62]:
# Effect of the importance-sampling proposal variance sigma^2 (n = 10000)
sigma2_values = [0.5, 1, 2, 4, 8]
n = 10000

for a in (2, 4):
    print(f"a = {a}, true P(Z>a) = {true_prob(a):.6e}")
    for sigma2 in sigma2_values:
        np.random.seed(42)
        (theta_is, lo_is, hi_is), var_is = importance_sampling(a, sigma2, n)
        print(f"  sigma^2={sigma2:>4}: theta_hat = {theta_is:.4e}, var = {var_is:.3e}, CI width = {hi_is - lo_is:.3e}")
    print()

a = 2, true P(Z>a) = 2.275013e-02
  sigma^2= 0.5: theta_hat = 2.2735e-02, var = 7.755e-04, CI width = 1.092e-03
  sigma^2=   1: theta_hat = 2.2755e-02, var = 1.214e-03, CI width = 1.366e-03
  sigma^2=   2: theta_hat = 2.2781e-02, var = 1.874e-03, CI width = 1.697e-03
  sigma^2=   4: theta_hat = 2.2788e-02, var = 2.830e-03, CI width = 2.086e-03
  sigma^2=   8: theta_hat = 2.2778e-02, var = 4.207e-03, CI width = 2.543e-03

a = 4, true P(Z>a) = 3.167124e-05
  sigma^2= 0.5: theta_hat = 3.1694e-05, var = 2.971e-09, CI width = 2.137e-06
  sigma^2=   1: theta_hat = 3.1718e-05, var = 4.535e-09, CI width = 2.640e-06
  sigma^2=   2: theta_hat = 3.1725e-05, var = 6.792e-09, CI width = 3.231e-06
  sigma^2=   4: theta_hat = 3.1740e-05, var = 1.005e-08, CI width = 3.929e-06
  sigma^2=   8: theta_hat = 3.1820e-05, var = 1.474e-08, CI width = 4.759e-06



### Discussion: efficiency of crude MC vs. importance sampling

- **Crude MC is essentially useless for rare events.** For $a=2$ ($\theta\approx 0.0228$) the crude estimator is unbiased and reasonably precise once $n\gtrsim 10^4$, but for $a=4$ ($\theta\approx 3.17\times10^{-5}$) a sample of $n=10^4$ contains *zero* hits with seed 42 -- the estimator returns exactly $0$ with a degenerate $(0,0)$ CI. Even at $n=10^5$ only a single hit was observed, giving $\hat\theta=10^{-5}$ with a 95% CI almost as wide as the estimate itself ($\approx 3.9\times10^{-5}$). Reliably resolving $\theta(4)$ with crude MC would require $n$ in the tens of millions.

- **Importance sampling with proposal $N(a,\sigma^2=1)$ is unbiased and dramatically more precise.** For $a=2$ it cuts the variance by **~94.6%** relative to crude MC at $n=10^4$ (CI width $1.4\times10^{-3}$ vs $6.0\times10^{-3}$). For $a=4$ it produces a tight, non-degenerate estimate ($\hat\theta\approx3.11\times10^{-5}$, CI $\approx(2.98,3.24)\times10^{-5}$) even though the crude estimator returns nothing useful at that sample size -- and IS already gives a sensible estimate at $n=100$, where crude MC sees zero hits for *both* $a=2$ and $a=4$.

- **Sample size**: increasing $n$ shrinks both methods' CIs at roughly the $1/\sqrt n$ rate, but crude MC needs $n\gg 1/\theta(a)$ before it produces *any* signal, whereas IS is informative even at very small $n$ because every draw lands near $\{Y>a\}$ by construction.

- **Effect of $\sigma^2$**: over $\sigma^2\in\{0.5,1,2,4,8\}$ the IS variance *increases monotonically* with $\sigma^2$ for both $a=2$ and $a=4$ -- $\sigma^2=0.5$ gave the smallest variance (e.g. $7.8\times10^{-4}$ vs $4.2\times10^{-3}$ at $\sigma^2=8$ for $a=2$). Intuitively, the zero-variance proposal is the law of $Z\mid Z>a$ (a truncated normal with small spread for large $a$), so a *narrower* proposal centred at $a$ tracks that conditional shape better than a wide one; an overly diffuse proposal wastes draws far from $a$ and produces large likelihood-ratio weights that inflate the variance. The point estimate itself stays close to the true value for all $\sigma^2$ tested -- IS remains unbiased regardless of $\sigma^2$, only its *variance* (and hence efficiency) depends on the choice.

- **Overall**: importance sampling is far more *efficient* -- for the same number of samples (i.e. roughly the same computational cost, since drawing from $N(a,\sigma^2)$ and evaluating the likelihood ratio costs about the same as drawing from $N(0,1)$) it achieves a much smaller variance, and unlike crude MC it remains usable when $\theta(a)$ is so small that crude sampling essentially never observes the event of interest.